# Alfa Connectome: first look at the data

This notebook follows the starter pack's quick start: each Parquet row group is one complete 20,000-row sequence. It reads one sequence for inspection, then samples complete sequences across train and validation for bounded-memory EDA. Sequence identifiers are shuffled IDs, not time features.

**Kernel:** Python (Alfa Connectome). Run from the repository root. The final cell writes `reports/eda.md`, `reports/eda.json`, and `reports/feature_stats.csv`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / 'datasets').exists():
    ROOT = ROOT.parent
assert (ROOT / 'datasets' / 'train.parquet').exists(), 'Open this notebook from the project or notebooks folder.'
sys.path.insert(0, str(ROOT))
print('Python:', sys.version.split()[0], '| PyArrow:', pq.__version__ if hasattr(pq, '__version__') else 'installed')
print('Project:', ROOT)

## 1. Inventory and schema

The row and row-group counts below come from the complete Parquet footers. This does not load the full dataset into RAM.

In [ ]:
paths = {name: ROOT / 'datasets' / f'{name}.parquet' for name in ('train', 'valid', 'valid_mask')}
files = {name: pq.ParquetFile(path) for name, path in paths.items()}
inventory = pd.DataFrame([
    {'file': name, 'size_GiB': path.stat().st_size / 2**30,
     'rows': files[name].metadata.num_rows,
     'sequences': files[name].metadata.num_row_groups,
     'columns': len(files[name].schema_arrow.names)}
    for name, path in paths.items()
])
display(inventory)
print('Train feature columns:', len(files['train'].schema_arrow.names) - 5)
print('First columns:', files['train'].schema_arrow.names[:8])
print('Last columns:', files['train'].schema_arrow.names[-8:])

## 2. Load one sequence

The first row group should have 20,000 ordered steps. Only steps 99 onward require a prediction.

In [ ]:
first = files['train'].read_row_group(0)
assert first.num_rows == 20_000
preview_columns = ['seq_ix', 'step_in_seq', 'need_prediction',
                   'i0_p0', 'i0_v0', 'i1_p0', 'a0', 't0', 't1']
display(first.select(preview_columns).slice(0, 6).to_pandas())
print('Shape:', first.shape)
print('Prediction rows:', int(first.column('need_prediction').to_numpy().sum()))
print('Steps:', first.column('step_in_seq')[0].as_py(), 'through',
      first.column('step_in_seq')[-1].as_py())

## 3. Sample complete sequences across each split

The helper in `eda.py` streams row groups one at a time and keeps only summary statistics. Change the two counts below for a broader sample. Validation target summaries use rows where both `need_prediction` and `is_scored` are true.

In [ ]:
from eda import scan_split, markdown_report

TRAIN_GROUPS = 64
VALID_GROUPS = 16
train = scan_split(paths['train'], TRAIN_GROUPS)
valid = scan_split(paths['valid'], VALID_GROUPS, paths['valid_mask'])
assert train['feature_count'] == valid['feature_count'] == 112
assert valid['mask_mismatch_groups'] == 0
checks = pd.DataFrame([
    {'split': name, 'sampled_sequences': len(result['sampled_group_indices']),
     'sampled_rows': result['sampled_rows'],
     'prediction_rows': result['sampled_prediction_rows'],
     'scored_rows': result['sampled_scored_rows'],
     'bad_sequence_groups': result['bad_sequence_groups'],
     'bad_step_groups': result['bad_step_groups']}
    for name, result in [('train', train), ('valid', valid)]
])
display(checks)
print('Validation mask mismatches:', valid['mask_mismatch_groups'])

## 4. Target distributions and temporal dependence

These are descriptive statistics on sampled rows. The competition score clips targets to [-2, 2] and uses Global Weighted Pearson, so the correlations shown here are not leaderboard scores.

In [ ]:
target_rows = []
for split_name, result in [('train', train), ('valid', valid)]:
    for target in ('t0', 't1'):
        item = result['targets'][target]
        target_rows.append({'split': split_name, 'target': target, 'n': item['count'],
                            'mean': item['mean'], 'std': item['std'],
                            **item['quantiles'],
                            'outside_clip_fraction': item['outside_score_clip_fraction'],
                            'lag1_correlation': item['lag1_correlation']})
display(pd.DataFrame(target_rows).round(4))
print('t0/t1 correlation:', {'train': train['target_correlation'],
                              'valid_scored': valid['target_correlation']})

In [ ]:
def sampled_targets(parquet, group_indices, scored=False):
    output = {'t0': [], 't1': []}
    columns = ['need_prediction', 't0', 't1'] + (['is_scored'] if scored else [])
    for index in group_indices:
        table = parquet.read_row_group(index, columns=columns)
        keep = table.column('need_prediction').to_numpy().astype(bool)
        if scored:
            keep &= table.column('is_scored').to_numpy().astype(bool)
        for target in output:
            output[target].append(table.column(target).to_numpy()[keep])
    return {target: np.concatenate(parts) for target, parts in output.items()}

train_y = sampled_targets(files['train'], train['sampled_group_indices'])
valid_y = sampled_targets(files['valid'], valid['sampled_group_indices'], scored=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), constrained_layout=True)
for ax, target in zip(axes, ('t0', 't1')):
    ax.hist(np.clip(train_y[target], -2, 2), bins=60, density=True,
            histtype='step', label='train prediction rows')
    ax.hist(np.clip(valid_y[target], -2, 2), bins=60, density=True,
            histtype='step', label='validation scored rows')
    ax.set(title=f'{target}: clipped to scoring range', xlabel='target', ylabel='density')
    ax.legend()
plt.show()

## 5. Feature quality, association, and split shift

Feature summaries use all rows in sampled sequences, including warm-up. Same-row feature/target correlations are exploratory; assess actual prediction quality with the competition's weighted metric on held-out sequences.

In [ ]:
feature_rows = [
    {'split': split_name, 'feature': feature, **stats}
    for split_name, result in [('train', train), ('valid', valid)]
    for feature, stats in result['features'].items()
]
feature_stats = pd.DataFrame(feature_rows)
display(feature_stats.sort_values(['nonfinite', 'zero_fraction'], ascending=False).head(20).round(4))
print('Total sampled nonfinite feature cells by split:')
display(feature_stats.groupby('split')['nonfinite'].sum())

In [ ]:
correlation_rows = [
    {'split': split_name, 'target': target, 'feature': feature, 'correlation': value}
    for split_name, result in [('train', train), ('valid', valid)]
    for target, pairs in result['feature_target_correlations'].items()
    for feature, value in pairs.items() if value is not None
]
correlations = pd.DataFrame(correlation_rows)
display(correlations.assign(abs_corr=correlations['correlation'].abs())
        .sort_values('abs_corr', ascending=False)
        .groupby(['split', 'target'], group_keys=False).head(10)
        .drop(columns='abs_corr').round(4))

In [ ]:
train_features = feature_stats.query("split == 'train'").set_index('feature')
valid_features = feature_stats.query("split == 'valid'").set_index('feature')
drift = pd.DataFrame({
    'train_mean': train_features['mean'],
    'valid_mean': valid_features['mean'],
    'train_std': train_features['std'],
})
drift['standardized_shift'] = (drift['valid_mean'] - drift['train_mean']) / drift['train_std']
drift = drift.replace([np.inf, -np.inf], np.nan).dropna(subset=['standardized_shift'])
top_drift = drift.reindex(drift['standardized_shift'].abs().sort_values(ascending=False).head(20).index)
display(top_drift.head(20).round(4))
top_drift['standardized_shift'].sort_values().plot.barh(figsize=(8, 6),
    title='Largest sampled train-to-validation mean shifts (train SD units)')
plt.xlabel('Standardized mean shift')
plt.tight_layout()
plt.show()

## 6. Save the report

This writes small, shareable summaries only. The raw Parquet data remains ignored by Git. If you change the sample sizes, rerun the notebook from the top before saving.

In [ ]:
mask_meta = files['valid_mask'].metadata
mask = {
    'file': 'datasets/valid_mask.parquet',
    'file_bytes': paths['valid_mask'].stat().st_size,
    'total_rows': mask_meta.num_rows,
    'row_groups': mask_meta.num_row_groups,
    'columns': files['valid_mask'].schema_arrow.names,
    'non_20000_row_groups': sum(mask_meta.row_group(i).num_rows != 20_000
                                for i in range(mask_meta.num_row_groups)),
}
assert valid['total_rows'] == mask['total_rows']
assert valid['row_groups'] == mask['row_groups']
def json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f'Cannot serialize {type(value).__name__}')


output = ROOT / 'reports'
output.mkdir(exist_ok=True)
(output / 'eda.json').write_text(json.dumps({'train': train, 'valid': valid,
                                            'valid_mask': mask}, indent=2, default=json_default), encoding='utf-8')
(output / 'eda.md').write_text(markdown_report(train, valid, mask), encoding='utf-8')
feature_stats.to_csv(output / 'feature_stats.csv', index=False)
print('Saved:', *(str(p) for p in output.iterdir()), sep='\n')

### Interpretation notes

The sample spans row groups, but it is still a sample. Sequence IDs must never be used as predictive features or chronological order. Do not use `is_scored` as a model input; it is unavailable at test time. The baseline in the starter pack reports 0.617052 Global Weighted Pearson on full validation, but this notebook does not run or reproduce that baseline score.